In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os
import json
import pickle
import random
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

SEEDS = [42, 123, 2024]

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

FEATURE_DIR = BASE_PROJECT / "processed_intra_features_hc_noaug"
OUT_DIR = BASE_PROJECT / "results_intra_handcrafted_svm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad"
]

ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL_TO_ID = {label: i for i, label in ID_TO_LABEL.items()}

print("FEATURE_DIR:", FEATURE_DIR)
print("OUT_DIR    :", OUT_DIR)

for ds in DATASETS:
    print(ds, (FEATURE_DIR / ds).exists())

FEATURE_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug
OUT_DIR    : /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_svm
emodb True
ravdess True
resd True


In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


def load_dataset_features(dataset_name):
    ds_dir = FEATURE_DIR / dataset_name

    X_train = np.load(ds_dir / "X_hc_train.npy")
    y_train = np.load(ds_dir / "y_train.npy")

    X_val = np.load(ds_dir / "X_hc_val.npy")
    y_val = np.load(ds_dir / "y_val.npy")

    X_test = np.load(ds_dir / "X_hc_test.npy")
    y_test = np.load(ds_dir / "y_test.npy")

    meta_train = pd.read_csv(ds_dir / "meta_train.csv")
    meta_val = pd.read_csv(ds_dir / "meta_val.csv")
    meta_test = pd.read_csv(ds_dir / "meta_test.csv")

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
    }


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def save_confusion_matrix_csv(cm, out_path):
    df_cm = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    df_cm.to_csv(out_path, index=True)


def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        labels=list(range(len(LABELS))),
        zero_division=0,
        output_dict=True
    )
    return pd.DataFrame(report).transpose()

In [6]:
def train_eval_svm_rbf(dataset_name, seed):
    set_seed(seed)

    data = load_dataset_features(dataset_name)

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_val = data["X_val"]
    y_val = data["y_val"]

    X_test = data["X_test"]
    y_test = data["y_test"]

    model = SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=seed
    )

    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    val_metrics = compute_metrics(y_val, y_val_pred)
    test_metrics = compute_metrics(y_test, y_test_pred)

    # Save per-run detail
    run_dir = OUT_DIR / dataset_name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "svm_model.pkl", "wb") as f:
        pickle.dump(model, f)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        **val_metrics
    }]).to_csv(run_dir / "val_metrics.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        **test_metrics
    }]).to_csv(run_dir / "test_metrics.csv", index=False)

    # Classification report
    make_report_df(y_val, y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(y_test, y_test_pred).to_csv(run_dir / "test_classification_report.csv")

    # Confusion matrix
    cm_val = confusion_matrix(y_val, y_val_pred, labels=list(range(len(LABELS))))
    cm_test = confusion_matrix(y_test, y_test_pred, labels=list(range(len(LABELS))))

    save_confusion_matrix_csv(cm_val, run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(cm_test, run_dir / "test_confusion_matrix.csv")

    # Predictions
    pred_val_df = data["meta_val"].copy()
    pred_val_df["y_true"] = y_val
    pred_val_df["y_pred"] = y_val_pred
    pred_val_df["true_label"] = [ID_TO_LABEL[i] for i in y_val]
    pred_val_df["pred_label"] = [ID_TO_LABEL[i] for i in y_val_pred]
    pred_val_df.to_csv(run_dir / "val_predictions.csv", index=False)

    pred_test_df = data["meta_test"].copy()
    pred_test_df["y_true"] = y_test
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in y_test]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]
    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        **val_metrics
    }

    row_test = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        **test_metrics
    }

    return row_val, row_test

In [7]:
all_rows = []

for dataset_name in DATASETS:
    print("=" * 90)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 90)

    for seed in SEEDS:
        print(f"Training SVM RBF | dataset={dataset_name} | seed={seed}")

        row_val, row_test = train_eval_svm_rbf(
            dataset_name=dataset_name,
            seed=seed
        )

        all_rows.append(row_val)
        all_rows.append(row_test)

        print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
        print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

results = pd.DataFrame(all_rows)
results.to_csv(OUT_DIR / "all_seed_results.csv", index=False)

display(results)
print("Saved:", OUT_DIR / "all_seed_results.csv")

DATASET: EMODB
Training SVM RBF | dataset=emodb | seed=42
VAL : {'accuracy': 0.5915, 'macro_f1': 0.5703, 'weighted_f1': 0.5712, 'uar': 0.5967}
TEST: {'accuracy': 0.7651, 'macro_f1': 0.7708, 'weighted_f1': 0.7666, 'uar': 0.7719}
Training SVM RBF | dataset=emodb | seed=123
VAL : {'accuracy': 0.5915, 'macro_f1': 0.5703, 'weighted_f1': 0.5712, 'uar': 0.5967}
TEST: {'accuracy': 0.7651, 'macro_f1': 0.7708, 'weighted_f1': 0.7666, 'uar': 0.7719}
Training SVM RBF | dataset=emodb | seed=2024
VAL : {'accuracy': 0.5915, 'macro_f1': 0.5703, 'weighted_f1': 0.5712, 'uar': 0.5967}
TEST: {'accuracy': 0.7651, 'macro_f1': 0.7708, 'weighted_f1': 0.7666, 'uar': 0.7719}
DATASET: RAVDESS
Training SVM RBF | dataset=ravdess | seed=42
VAL : {'accuracy': 0.4148, 'macro_f1': 0.4051, 'weighted_f1': 0.4021, 'uar': 0.4167}
TEST: {'accuracy': 0.5852, 'macro_f1': 0.5626, 'weighted_f1': 0.5752, 'uar': 0.5729}
Training SVM RBF | dataset=ravdess | seed=123
VAL : {'accuracy': 0.4148, 'macro_f1': 0.4051, 'weighted_f1': 0.4

,dataset,seed,split,accuracy,macro_f1,weighted_f1,uar
0,emodb,42,val,0.591549,0.570281,0.571169,0.596737
1,emodb,42,test,0.765101,0.770845,0.766550,0.771862
2,emodb,123,val,0.591549,0.570281,0.571169,0.596737
3,emodb,123,test,0.765101,0.770845,0.766550,0.771862
4,emodb,2024,val,0.591549,0.570281,0.571169,0.596737
5,emodb,2024,test,0.765101,0.770845,0.766550,0.771862
6,ravdess,42,val,0.414773,0.405081,0.402133,0.416667
7,ravdess,42,test,0.585227,0.562632,0.575213,0.572917
8,ravdess,123,val,0.414773,0.405081,0.402133,0.416667
9,ravdess,123,test,0.585227,0.562632,0.575213,0.572917


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_svm/all_seed_results.csv


In [8]:
metrics = ["accuracy", "macro_f1", "weighted_f1", "uar"]

summary_rows = []

for dataset_name in DATASETS:
    for split in ["val", "test"]:
        sub = results[
            (results["dataset"] == dataset_name) &
            (results["split"] == split)
        ]

        row = {
            "dataset": dataset_name,
            "split": split,
            "n_seeds": len(sub)
        }

        for metric in metrics:
            row[f"{metric}_mean"] = sub[metric].mean()
            row[f"{metric}_std"] = sub[metric].std(ddof=1)

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "summary_mean_std.csv", index=False)

display(summary)
print("Saved:", OUT_DIR / "summary_mean_std.csv")

,dataset,split,n_seeds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,uar_mean,uar_std
0,emodb,val,3,0.591549,0.000000e+00,0.570281,0.0,0.571169,0.000000e+00,0.596737,0.0
1,emodb,test,3,0.765101,0.000000e+00,0.770845,0.0,0.766550,0.000000e+00,0.771862,0.0
2,ravdess,val,3,0.414773,0.000000e+00,0.405081,0.0,0.402133,6.798700e-17,0.416667,0.0
3,ravdess,test,3,0.585227,0.000000e+00,0.562632,0.0,0.575213,0.000000e+00,0.572917,0.0
4,resd,val,3,0.188172,3.399350e-17,0.175361,0.0,0.176799,0.000000e+00,0.183854,0.0
5,resd,test,3,0.230216,0.000000e+00,0.177384,0.0,0.201610,0.000000e+00,0.216041,0.0


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_svm/summary_mean_std.csv


In [9]:
def mean_std_str(mean, std, scale=100):
    return f"{mean * scale:.2f} ± {std * scale:.2f}"


paper_rows = []

for dataset_name in DATASETS:
    sub = summary[
        (summary["dataset"] == dataset_name) &
        (summary["split"] == "test")
    ].iloc[0]

    paper_rows.append({
        "Dataset": dataset_name.upper(),
        "Accuracy": mean_std_str(sub["accuracy_mean"], sub["accuracy_std"]),
        "UAR": mean_std_str(sub["uar_mean"], sub["uar_std"]),
        "Macro-F1": mean_std_str(sub["macro_f1_mean"], sub["macro_f1_std"]),
        "Weighted-F1": mean_std_str(sub["weighted_f1_mean"], sub["weighted_f1_std"]),
    })

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUT_DIR / "paper_table_test_mean_std.csv", index=False)

display(paper_table)
print("Saved:", OUT_DIR / "paper_table_test_mean_std.csv")

,Dataset,Accuracy,UAR,Macro-F1,Weighted-F1
0,EMODB,76.51 ± 0.00,77.19 ± 0.00,77.08 ± 0.00,76.66 ± 0.00
1,RAVDESS,58.52 ± 0.00,57.29 ± 0.00,56.26 ± 0.00,57.52 ± 0.00
2,RESD,23.02 ± 0.00,21.60 ± 0.00,17.74 ± 0.00,20.16 ± 0.00


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_svm/paper_table_test_mean_std.csv


In [10]:
per_class_rows = []

for dataset_name in DATASETS:
    for seed in SEEDS:
        report_path = OUT_DIR / dataset_name / f"seed_{seed}" / "test_classification_report.csv"
        report = pd.read_csv(report_path, index_col=0)

        for label in LABELS:
            per_class_rows.append({
                "dataset": dataset_name,
                "seed": seed,
                "class": label,
                "precision": report.loc[label, "precision"],
                "recall": report.loc[label, "recall"],
                "f1": report.loc[label, "f1-score"],
                "support": report.loc[label, "support"],
            })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(OUT_DIR / "per_class_test_all_seeds.csv", index=False)

per_class_summary = (
    per_class_df
    .groupby(["dataset", "class"])
    .agg(
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        support_mean=("support", "mean"),
    )
    .reset_index()
)

per_class_summary.to_csv(OUT_DIR / "per_class_test_summary_mean_std.csv", index=False)

display(per_class_summary)
print("Saved:", OUT_DIR / "per_class_test_summary_mean_std.csv")

,dataset,class,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support_mean
0,emodb,angry,0.708333,0.0,0.586207,0.0,0.641509,0.0,29.0
1,emodb,disgust,0.850000,0.0,0.772727,0.0,0.809524,0.0,22.0
2,emodb,fear,0.700000,0.0,0.840000,0.0,0.763636,0.0,25.0
3,emodb,happy,0.586207,0.0,0.708333,0.0,0.641509,0.0,24.0
4,emodb,neutral,0.869565,0.0,0.909091,0.0,0.888889,0.0,22.0
5,emodb,sad,0.956522,0.0,0.814815,0.0,0.880000,0.0,27.0
6,ravdess,angry,0.657143,0.0,0.718750,0.0,0.686567,0.0,32.0
7,ravdess,disgust,0.615385,0.0,0.750000,0.0,0.676056,0.0,32.0
8,ravdess,fear,0.625000,0.0,0.468750,0.0,0.535714,0.0,32.0
9,ravdess,happy,0.560976,0.0,0.718750,0.0,0.630137,0.0,32.0


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_handcrafted_svm/per_class_test_summary_mean_std.csv


In [11]:
for dataset_name in DATASETS:
    cms = []

    for seed in SEEDS:
        cm_path = OUT_DIR / dataset_name / f"seed_{seed}" / "test_confusion_matrix.csv"
        cm = pd.read_csv(cm_path, index_col=0).values
        cms.append(cm)

    cm_mean = np.mean(cms, axis=0)
    cm_std = np.std(cms, axis=0, ddof=1)

    cm_mean_df = pd.DataFrame(cm_mean, index=LABELS, columns=LABELS)
    cm_std_df = pd.DataFrame(cm_std, index=LABELS, columns=LABELS)

    ds_out = OUT_DIR / dataset_name
    cm_mean_df.to_csv(ds_out / "test_confusion_matrix_mean.csv")
    cm_std_df.to_csv(ds_out / "test_confusion_matrix_std.csv")

    print("=" * 80)
    print(dataset_name.upper())
    print("Mean confusion matrix:")
    display(cm_mean_df)

    print("Std confusion matrix:")
    display(cm_std_df)

EMODB
Mean confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,17.0,0.0,1.0,11.0,0.0,0.0
disgust,0.0,17.0,4.0,0.0,1.0,0.0
fear,0.0,1.0,21.0,1.0,1.0,1.0
happy,7.0,0.0,0.0,17.0,0.0,0.0
neutral,0.0,1.0,1.0,0.0,20.0,0.0
sad,0.0,1.0,3.0,0.0,1.0,22.0


Std confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,0.0,0.0,0.0,0.0,0.0,0.0
disgust,0.0,0.0,0.0,0.0,0.0,0.0
fear,0.0,0.0,0.0,0.0,0.0,0.0
happy,0.0,0.0,0.0,0.0,0.0,0.0
neutral,0.0,0.0,0.0,0.0,0.0,0.0
sad,0.0,0.0,0.0,0.0,0.0,0.0


RAVDESS
Mean confusion matrix:


...

,angry,disgust,fear,happy,neutral,sad
angry,0.0,0.0,0.0,0.0,0.0,0.0
disgust,0.0,0.0,0.0,0.0,0.0,0.0
fear,0.0,0.0,0.0,0.0,0.0,0.0
happy,0.0,0.0,0.0,0.0,0.0,0.0
neutral,0.0,0.0,0.0,0.0,0.0,0.0
sad,0.0,0.0,0.0,0.0,0.0,0.0


RESD
Mean confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,15.0,6.0,5.0,6.0,0.0,2.0
disgust,0.0,5.0,3.0,3.0,3.0,1.0
fear,3.0,2.0,0.0,20.0,4.0,0.0
happy,2.0,5.0,2.0,12.0,2.0,0.0
neutral,5.0,5.0,6.0,3.0,0.0,1.0
sad,3.0,7.0,4.0,0.0,4.0,0.0


Std confusion matrix:


,angry,disgust,fear,happy,neutral,sad
angry,0.0,0.0,0.0,0.0,0.0,0.0
disgust,0.0,0.0,0.0,0.0,0.0,0.0
fear,0.0,0.0,0.0,0.0,0.0,0.0
happy,0.0,0.0,0.0,0.0,0.0,0.0
neutral,0.0,0.0,0.0,0.0,0.0,0.0
sad,0.0,0.0,0.0,0.0,0.0,0.0
